In [ ]:
import xarray as xr 
import numpy as np
import json
from datetime import date, timedelta
import os
import zarr

In [5]:
ds_binned_all.to_zarr('../data/trial.zarr', mode='w')

In [12]:
z = zarr.open('../data/trial.zarr', mode='r')
z.keys()

KeysView(<zarr.hierarchy.Group '/' read-only>)

In [15]:
print(z['lat'].shape, z['lon'].shape, z['wavelength'].shape, z['spectral_radiance'].shape, 
      z['spectral_radiance_std'].shape, )

(720,) (1440,) (63,) (720, 1440, 63) (720, 1440, 63)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
ds_binned_all = xr.open_dataset('../data/2512RAD.nc')

# ── select nearest grid point to 65°N, 130°W ─────────────────────────────────
point = ds_binned_all.sel(lat=65.0, lon=-130.0, method='nearest')

wl  = point['wavelength'].values
rad = point['spectral_radiance'].values
std = point['spectral_radiance_std'].values
cnt = point['spectral_radiance_count'].values

# drop spectral channels with no wavelength label OR no radiance
valid         = np.isfinite(wl) & np.isfinite(rad)
valid_indices = np.where(valid)[0]
wl_v, rad_v, std_v, cnt_v = wl[valid], rad[valid], std[valid], cnt[valid]

orig_to_filt = {orig: i for i, orig in enumerate(valid_indices)}

# ── constants ────────────────────────────────────────────────────────────────
H, C, KB = 6.626e-34, 2.998e8, 1.381e-23

def planck_um(lam_um, T):
    """Spectral radiance in W/m²/sr/μm for wavelength in μm."""
    lam = lam_um * 1e-6
    return (2 * H * C**2 / lam**5) / np.expm1(H * C / (lam * KB * T)) * 1e-6

def brightness_temp_um(lam_um, B):
    """Invert Planck. lam_um in μm, B in W/m²/sr/μm. Returns T in K."""
    lam = lam_um * 1e-6
    B_si = B * 1e6                              # → W/m²/sr/m
    # guard against non-positive radiance
    if not np.isfinite(B_si) or B_si <= 0:
        return np.nan
    return (H * C / (lam * KB)) / np.log1p(2 * H * C**2 / (lam**5 * B_si))

# ── find peak-radiance channel and invert Planck ─────────────────────────────
peak_i      = int(np.nanargmax(rad_v))
peak_lam_um = wl_v[peak_i]
peak_rad    = rad_v[peak_i]
T_peak      = brightness_temp_um(peak_lam_um, peak_rad)

print(f'Peak channel: λ = {peak_lam_um:.3f} μm, '
      f'L = {peak_rad:.3f} W/m²/sr/μm, T_b = {T_peak:.2f} K')

# Reference Planck curve uses the inverted temperature (not 280 K)
T_REF    = T_peak
planck_v = planck_um(wl_v, T_REF)

# ── matplotlib figure (unchanged structure, T_REF now derived) ───────────────
HIGHLIGHT_IDX   = [12, 24, 30, 32]
HIGHLIGHT_FILT  = [orig_to_filt[ch] for ch in HIGHLIGHT_IDX if ch in orig_to_filt]
HIGHLIGHT_COLOR = 'crimson'

def make_colors(n, highlight_filt, default_color):
    colors = [default_color] * n
    for i in highlight_filt:
        colors[i] = HIGHLIGHT_COLOR
    return colors

# per-bar colors using the filtered positions (accounts for removed NaN channels)
hl = set(HIGHLIGHT_FILT)
c_rad = ['crimson' if i in hl else 'darkorange'      for i in range(len(wl_v))]
c_std = ['crimson' if i in hl else 'deepskyblue'     for i in range(len(wl_v))]
c_cnt = ['crimson' if i in hl else 'mediumseagreen' for i in range(len(wl_v))]

# customdata per bar: [original_channel_index, wavelength]
hover_data = np.stack([valid_indices.astype(float), wl_v], axis=1)

forward_diff = np.diff(wl_v)  # length N-1, distance to next point
backward_diff = forward_diff   # same array, but interpreted as distance from previous point

# For interior points, take the minimum of forward and backward gaps
bar_widths = np.empty_like(wl_v)
bar_widths[0] = forward_diff[0]              # first point: only forward exists
bar_widths[-1] = backward_diff[-1]           # last point: only backward exists
bar_widths[1:-1] = np.maximum(np.minimum(forward_diff[1:], 1.2), np.minimum(backward_diff[:-1], 1.2))*0.95
bar_widths = np.append(bar_widths, bar_widths[-1])

fig = make_subplots(
    rows=2, cols=1,
    specs=[[{'secondary_y': True}], [{'secondary_y': False}]], #, [{'secondary_y': False}]]
    shared_xaxes=True,
    subplot_titles=('Spectral Radiance', 'Std of Spectral Radiance', 'Observation Count'),
    vertical_spacing=0.07,
)

# ── row 1: radiance bars + full Planck line ───────────────────────────────────
lam_dense  = np.linspace(wl_v.min(), wl_v.max(), 500)
planck_dense = planck_um(lam_dense, T_peak)

masking = np.array([True if i in HIGHLIGHT_FILT else False for i in range(len(bar_widths)-1)])

fig.add_trace(go.Bar(
    x=wl_v[masking], y=rad_v[masking],
    width=bar_widths[:-1][masking],
    name='Channels available on maps',
    marker_color='crimson',
    customdata=hover_data[masking],
    hovertemplate='Channel %{customdata[0]:.0f}<br>λ = %{x:.3f} μm<br>L = %{y:.3f} W/m²/sr/μm<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=wl_v[~masking], y=rad_v[~masking],
    width=bar_widths[:-1][~masking],
    name='Other channels',
    marker_color='darkorange',
    customdata=hover_data[~masking],
    hovertemplate='Channel %{customdata[0]:.0f}<br>λ = %{x:.3f} μm<br>L = %{y:.3f} W/m²/sr/μm<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=lam_dense, y=planck_dense,
    mode='lines',
    name=f'Planck B(λ, {T_peak:.1f} K)',
    line=dict(color='tomato', dash='dash', width=2),
    hovertemplate='λ = %{x:.3f} μm<br>B = %{y:.3f} W/m²/sr/μm<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[peak_lam_um], y=[peak_rad],
    mode='markers',
    name=f'Peak: {peak_lam_um:.2f} μm, T_b = {T_peak:.1f} K',
    marker=dict(color='crimson', size=12, symbol='star',
                line=dict(color='black', width=1)),
    hovertemplate=(f'Peak channel<br>λ = {peak_lam_um:.3f} μm<br>'
                   f'L = {peak_rad:.3f} W/m²/sr/μm<br>'
                   f'T_b = {T_peak:.2f} K<extra></extra>'),
), row=1, col=1)



# ── row 2: std bars ───────────────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=wl_v, y=std_v,
    marker_color=c_std,
    width=0.8,
    name='Std',
    showlegend=False,
    customdata=hover_data,
    hovertemplate=(
        '<b>Channel %{customdata[0]:.0f}</b><br>'
        'Wavelength: %{customdata[1]:.4f} μm<br>'
        'Std: %{y:.4e}<extra></extra>'
    ),
), row=2, col=1)

# # ── row 3: count bars ─────────────────────────────────────────────────────────
# fig.add_trace(go.Bar(
#     x=wl_v, y=cnt_v,
#     marker_color=c_cnt,
#     name='Count',
#     showlegend=False,
#     customdata=hover_data,
#     hovertemplate=(
#         '<b>Channel %{customdata[0]:.0f}</b><br>'
#         'Wavelength: %{customdata[1]:.4f} μm<br>'
#         'Count: %{y:d}<extra></extra>'
#     ),
# ), row=3, col=1)

# ── axes ──────────────────────────────────────────────────────────────────────
fig.update_yaxes(title_text=f'Spectral radiance', row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text=f'Planck B(λ,{T_REF}K) (W/m²/sr/μm)',
                 title_font_color='tomato', tickfont_color='tomato',
                 row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text='Std of Spectral Radiance', row=2, col=1)
# fig.update_yaxes(title_text='Observation Count',        row=3, col=1)
fig.update_xaxes(title_text='Wavelength (μm)',          row=2, col=1)

fig.update_layout(
    title=dict(text = f'Spectral Radiance & Planck fit at {point.lat.item():.2f}°N, {abs(point.lon.item()):.2f}°W',
                    x=0.06, y=1, yanchor='top'),
    margin=dict(l=0, r=0, t=40, b=40),
    bargap=0,
    legend=dict(
        orientation='v',
        x=0.94, y=0.88,
        xanchor='right', yanchor='bottom',
        entrywidth=180,           # px per entry; tune to taste
        entrywidthmode='pixels',  # or 'fraction' if you prefer 0–1
    ),
)

fig.show()
fig.write_html('../data/trial1.html')

Peak channel: λ = 11.406 μm, L = 4.190 W/m²/sr/μm, T_b = 252.28 K


In [56]:
bar_widths[:-1][~masking]

array([0.34375   , 0.475     , 0.7050781 , 1.14      , 1.14      ,
       0.81640625, 0.81640625, 0.7792969 , 0.7792969 , 1.14      ,
       1.14      , 0.7792969 , 0.80898434, 0.80898434, 0.8015625 ,
       0.80898434, 0.80898434, 0.80898434, 0.79414064, 0.79414064,
       0.79414064, 0.8238281 , 1.14      , 1.14      , 0.6828125 ,
       0.75703126, 0.83867186, 0.83867186, 0.81640625, 0.77187496,
       0.7792969 , 0.83867186, 0.83867186, 0.83125   , 0.83125   ,
       0.83867186, 0.83867186, 0.8683594 , 0.8683594 , 0.95742184,
       0.95742184, 1.009375  , 1.009375  , 0.9128906 , 0.9128906 ,
       0.81640625, 0.9796875 , 0.9796875 , 0.7347656 , 0.7578125 ],
      dtype=float32)

In [13]:
forward_diff = np.diff(wl_v)  # length N-1, distance to next point
backward_diff = forward_diff   # same array, but interpreted as distance from previous point

# For interior points, take the minimum of forward and backward gaps
bar_widths = np.empty_like(wl_v)
bar_widths[0] = forward_diff[0]              # first point: only forward exists
bar_widths[-1] = backward_diff[-1]           # last point: only backward exists
bar_widths[1:-1] = np.minimum(forward_diff[1:], backward_diff[:-1])

In [32]:
masking

array([ True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True, False,
        True,  True,  True,  True,  True, False,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True])

In [12]:
wl

array([       nan,        nan,        nan,  3.921875 ,  4.265625 ,
        4.765625 ,  5.5078125,        nan,        nan,  8.3984375,
        8.8984375,  9.7578125, 10.5859375, 11.40625  , 12.2265625,
       13.0390625,        nan,        nan, 15.9609375, 16.507812 ,
       17.328125 , 18.179688 , 19.023438 , 19.859375 , 20.710938 ,
       21.539062 , 22.390625 , 23.21875  , 24.054688 , 24.890625 ,
       25.703125 , 26.539062 , 27.40625  , 28.226562 ,        nan,
              nan, 31.15625  , 31.835938 , 32.554688 , 33.351562 ,
       34.234375 , 35.09375  , 35.90625  , 36.6875   , 37.507812 ,
       38.390625 , 39.25     , 40.125    , 40.929688 , 41.8125   ,
       42.554688 , 43.46875  , 44.1875   , 45.195312 , 45.875    ,
       46.9375   , 47.609375 , 48.570312 , 49.429688 , 50.09375  ,
       51.125    , 51.898438 , 52.65625  ], dtype=float32)